## 1. Imports & Config

In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

RAW_DIR = Path('../data/raw')          # <-- adjust to wherever your CSVs live
OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    'drivers':               'drivers.csv',
    'constructors':          'constructors.csv',
    'circuits':              'circuits.csv',
    'races':                 'races.csv',
    'results':               'results.csv',
    'qualifying':            'qualifying.csv',
    'pit_stops':             'pit_stops.csv',
    'lap_times':             'lap_times.csv',
    'status':                'status.csv',
}


## 2. Load Raw Data

In [5]:
raw = {
    name: pd.read_csv(RAW_DIR / fname)
    for name, fname in RAW_FILES.items()
}

# Sanity check
for name, df in raw.items():
    print(f"{name:15s} -> {df.shape[0]:>7,} rows, {df.shape[1]:>2} cols")


drivers         ->     865 rows,  9 cols
constructors    ->     214 rows,  5 cols
circuits        ->      78 rows,  9 cols
races           ->   1,171 rows, 18 cols
results         ->  27,414 rows, 18 cols
qualifying      ->  11,146 rows,  9 cols
pit_stops       ->  22,424 rows,  7 cols
lap_times       -> 875,093 rows,  6 cols
status          ->     140 rows,  2 cols


## 3. Profiling

In [6]:
def profile_table(df, name):
    print(f"=== {name} ===")
    print(df.info())
    print("\nNull counts:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    print(f"\nExact duplicate rows: {df.duplicated().sum()}")
    # F1 Kaggle CSVs commonly use the literal string '\\N' as a null placeholder
    obj_cols = df.select_dtypes(include='object').columns
    n_placeholder = (df[obj_cols] == r'\N').sum().sum()
    print(f"'\\N' placeholder count across object columns: {n_placeholder}")
    print("-" * 60)

for name, df in raw.items():
    profile_table(df, name)


=== drivers ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 865 entries, 0 to 864
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   driverId     865 non-null    int64 
 1   driverRef    865 non-null    object
 2   number       865 non-null    object
 3   code         865 non-null    object
 4   forename     865 non-null    object
 5   surname      865 non-null    object
 6   dob          865 non-null    object
 7   nationality  865 non-null    object
 8   url          865 non-null    object
dtypes: int64(1), object(8)
memory usage: 60.9+ KB
None

Null counts:
Series([], dtype: int64)

Exact duplicate rows: 0
'\N' placeholder count across object columns: 1559
------------------------------------------------------------
=== constructors ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          -------

In [7]:
# Standardize the '\\N' placeholder into a real NaN across every table,
# before any type conversion. This is the single most important cleaning step —
# leaving '\\N' as text silently breaks numeric conversions and joins downstream.

for name, df in raw.items():
    raw[name] = df.replace(r'\N', np.nan)


## 4. Shared Parsing Helpers

Reused across `qualifying`, `lap_times`, `pit_stops`, and `results` — matches the
`fnParseLapTime` custom function referenced in the Power Query documentation, validated
here first in a much easier debugging environment.


In [8]:
def parse_lap_time_to_ms(time_str):
    """Parses 'm:ss.SSS' (e.g. '1:23.456') lap/quali time strings into milliseconds.
    Returns np.nan for missing/unparseable values."""
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).strip()
    match = re.match(r'^(?:(\d+):)?(\d{1,2})\.(\d{1,3})$', time_str)
    if not match:
        return np.nan
    minutes = int(match.group(1)) if match.group(1) else 0
    seconds = int(match.group(2))
    millis_str = match.group(3).ljust(3, '0')
    millis = int(millis_str)
    return minutes * 60_000 + seconds * 1_000 + millis


def parse_duration_to_ms(duration_str):
    """Parses pit stop 'duration' strings (e.g. '23.456') into milliseconds."""
    if pd.isna(duration_str):
        return np.nan
    try:
        return round(float(duration_str) * 1000)
    except ValueError:
        return np.nan


def parse_gap_to_seconds(gap_str):
    """Parses race-result 'time' gap strings (e.g. '+5.291s', '+1 Lap') into seconds.
    Returns np.nan for leader's own time or non-gap statuses (handled via status join instead)."""
    if pd.isna(gap_str):
        return np.nan
    gap_str = str(gap_str).strip()
    match = re.match(r'^\+?(\d+\.\d+)s?$', gap_str)
    if match:
        return float(match.group(1))
    return np.nan  # e.g. '+1 Lap', or the race winner's absolute finish time — handled separately

# quick self-test
assert parse_lap_time_to_ms('1:23.456') == 83456
assert parse_duration_to_ms('23.456') == 23456
assert parse_gap_to_seconds('+5.291s') == 5.291
print("Parsing helpers OK")


Parsing helpers OK


## 5. Clean `status` — build `StatusCategory`

The ~130 raw status strings get grouped into a small set of categories

In [12]:
import re

def categorize_status(status_description):
    s = str(status_description).lower()

    if s == "finished" or re.match(r"^\+\d+ lap", s):
        return "Finished"

    accident_kw = [
        "accident", "collision", "spun off", "damage",
        "broken wing", "front wing", "rear wing", "wing",
        "fire", "heat shield fire", "debris"
    ]

    mechanical_kw = [
        "driveshaft", "track rod", "pneumatics", "power loss",
        "drivetrain", "ignition", "chassis", "stalled",
        "halfshaft", "crankshaft", "injection", "distributor",
        "cv joint", "spark plugs", "axle", "magneto",
        "supercharger", "power unit", "ers", "seat",
        "driver seat", "undertray", "cooling system",
        "vibration", "vibrations", "technical",
        "handling", "launch control"
    ]

    driver_kw = [
        "disqualified", "excluded",
        "did not qualify", "did not prequalify",
        "withdrew", "retired",
        "not classified", "not restarted",
        "107% rule", "underweight",
        "safety", "safety concerns", "safety belt",
        "injured", "injury", "eye injury",
        "illness", "physical", "driver unwell"
    ]

    if any(k in s for k in accident_kw):
        return "Accident"

    if any(k in s for k in mechanical_kw):
        return "Mechanical"

    if any(k in s for k in driver_kw):
        return "Driver/Admin"

    return "Other"

status_df = raw['status'].copy()
status_df['StatusCategory'] = status_df['status'].apply(categorize_status)

print(status_df['StatusCategory'].value_counts())
print("\nReview any 'Other' entries below and add keywords above as needed:")
print(status_df.loc[status_df['StatusCategory'] == 'Other', 'status'].tolist())


StatusCategory
Other           45
Finished        34
Mechanical      28
Driver/Admin    19
Accident        14
Name: count, dtype: int64

Review any 'Other' entries below and add keywords above as needed:
['Engine', 'Gearbox', 'Transmission', 'Clutch', 'Hydraulics', 'Electrical', 'Radiator', 'Suspension', 'Brakes', 'Differential', 'Overheating', 'Mechanical', 'Tyre', 'Puncture', 'Fuel pressure', 'Water pressure', 'Refuelling', 'Wheel', 'Throttle', 'Steering', 'Electronics', 'Exhaust', 'Oil leak', 'Wheel rim', 'Water leak', 'Fuel pump', 'Oil pressure', 'Tyre puncture', 'Out of fuel', 'Wheel nut', 'Wheel bearing', 'Fuel system', 'Oil line', 'Fuel rig', 'Fuel', 'Battery', 'Alternator', 'Oil pump', 'Fuel leak', 'Turbo', 'Water pump', 'Fuel pipe', 'Oil pipe', 'Water pipe', 'Brake duct']


## 6. Clean `drivers`

In [13]:
drivers_df = raw['drivers'].copy()

# Normalize name casing — historical entries are inconsistent (some all-caps)
drivers_df['forename'] = drivers_df['forename'].str.strip().str.title()
drivers_df['surname'] = drivers_df['surname'].str.strip().str.title()
drivers_df['FullName'] = drivers_df['forename'] + ' ' + drivers_df['surname']

# Flag estimated/missing DOB rather than dropping the row
drivers_df['IsDOBEstimated'] = drivers_df['dob'].isna()

# Driver code fallback for pre-2014 drivers without an official 3-letter code
def derive_driver_code(row):
    if pd.notna(row.get('code')) and row['code'] not in ('\\N', ''):
        return row['code'], False
    return row['surname'][:3].upper(), True

codes = drivers_df.apply(derive_driver_code, axis=1)
drivers_df['DriverCode'] = codes.apply(lambda x: x[0])
drivers_df['IsCodeDerived'] = codes.apply(lambda x: x[1])

drivers_df.head()


,driverId,driverRef,number,code,forename,surname,dob,nationality,url,FullName,IsDOBEstimated,DriverCode,IsCodeDerived
0,1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton,Lewis Hamilton,False,HAM,False
1,2,heidfeld,NaN,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld,Nick Heidfeld,False,HEI,False
2,3,rosberg,6,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg,Nico Rosberg,False,ROS,False
3,4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso,Fernando Alonso,False,ALO,False
4,5,kovalainen,NaN,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen,Heikki Kovalainen,False,KOV,False


## 7. Clean `circuits` & `races`

Merge races with circuits, derive `CircuitType` (manual mapping — extend against
your actual circuit list), and parse date/time into proper datetime + season/round.


In [15]:
circuits_df = raw['circuits'].copy()

# Manual circuit type classification
CIRCUIT_TYPE_MAP = {
    'albert_park': 'Hybrid',        'sepang': 'Permanent',         'bahrain': 'Permanent',
    'catalunya': 'Permanent',       'istanbul': 'Permanent',       'monaco': 'Street',
    'villeneuve': 'Hybrid',         'magny_cours': 'Permanent',    'silverstone': 'Permanent',
    'hockenheimring': 'Permanent',  'hungaroring': 'Permanent',    'valencia': 'Hybrid',
    'spa': 'Permanent',             'monza': 'Permanent',          'marina_bay': 'Street',
    'fuji': 'Permanent',            'shanghai': 'Permanent',       'interlagos': 'Permanent',
    'indianapolis': 'Permanent',    'nurburgring': 'Permanent',    'imola': 'Permanent',
    'suzuka': 'Permanent',          'yas_marina': 'Permanent',     'galvez': 'Permanent',
    'jerez': 'Permanent',           'estoril': 'Permanent',        'okayama': 'Permanent',
    'adelaide': 'Street',           'kyalami': 'Permanent',        'donington': 'Permanent',
    'rodriguez': 'Permanent',       'phoenix': 'Street',           'ricard': 'Permanent',
    'yeongam': 'Permanent',         'jacarepagua': 'Permanent',    'detroit': 'Street',
    'brands_hatch': 'Permanent',    'zandvoort': 'Permanent',      'zolder': 'Permanent',
    'dijon': 'Permanent',           'dallas': 'Street',            'long_beach': 'Street',
    'las_vegas': 'Street',          'jarama': 'Permanent',         'watkins_glen': 'Permanent',
    'anderstorp': 'Permanent',      'mosport': 'Permanent',        'montjuic': 'Street',
    'nivelles': 'Permanent',        'charade': 'Permanent',        'tremblant': 'Permanent',
    'essarts': 'Permanent',         'lemans': 'Permanent',         'reims': 'Permanent',
    'george': 'Permanent',          'zeltweg': 'Permanent',        'aintree': 'Permanent',
    'boavista': 'Street',           'riverside': 'Permanent',      'avus': 'Hybrid',
    'monsanto': 'Street',           'sebring': 'Permanent',        'ain-diab': 'Street',
    'pescara': 'Street',            'bremgarten': 'Street',        'pedralbes': 'Street',
    'buddh': 'Permanent',           'americas': 'Permanent',       'red_bull_ring': 'Permanent',
    'sochi': 'Street',              'baku': 'Street',              'portimao': 'Permanent',
    'mugello': 'Permanent',         'jeddah': 'Street',            'losail': 'Permanent',
    'miami': 'Hybrid',              'vegas': 'Street',             'madring': 'Street'
}
circuits_df['CircuitType'] = circuits_df['circuitRef'].map(CIRCUIT_TYPE_MAP).fillna('Unclassified')

races_df = raw['races'].copy()
races_df['race_date'] = pd.to_datetime(races_df['date'], errors='coerce')
races_df['Season'] = races_df['year']
races_df = races_df.rename(columns={'round': 'Round', 'name': 'RaceName'})

races_enriched = races_df.merge(
    circuits_df[['circuitId', 'name', 'country', 'location', 'CircuitType']],
    on='circuitId', how='left', suffixes=('', '_circuit')
).rename(columns={'name_circuit': 'CircuitName'} if 'name_circuit' in races_df.columns else {})

races_enriched.head()


,raceId,year,Round,circuitId,RaceName,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,race_date,Season,name,country,location,CircuitType
0,1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,http://en.wikipedia.org/wiki/2009_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-03-29,2009,Albert Park Grand Prix Circuit,Australia,Melbourne,Hybrid
1,2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-04-05,2009,Sepang International Circuit,Malaysia,Kuala Lumpur,Permanent
2,3,2009,3,17,Chinese Grand Prix,2009-04-19,07:00:00,http://en.wikipedia.org/wiki/2009_Chinese_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-04-19,2009,Shanghai International Circuit,China,Shanghai,Permanent
3,4,2009,4,3,Bahrain Grand Prix,2009-04-26,12:00:00,http://en.wikipedia.org/wiki/2009_Bahrain_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-04-26,2009,Bahrain International Circuit,Bahrain,Sakhir,Permanent
4,5,2009,5,4,Spanish Grand Prix,2009-05-10,12:00:00,http://en.wikipedia.org/wiki/2009_Spanish_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-05-10,2009,Circuit de Barcelona-Catalunya,Spain,Montmeló,Permanent


## 8. Clean `results`

Dedupe, replace remaining placeholders, split the gap string, and derive the core
flags (`PositionsGained`, `IsDNF`, `IsPodium`, `IsWin`) used throughout the DAX layer.


In [16]:
results_df = raw['results'].copy()

# Remove duplicate rows on the natural key (rare, but present from merge artifacts)
before = len(results_df)
results_df = results_df.drop_duplicates(subset=['raceId', 'driverId'])
print(f"Dropped {before - len(results_df)} duplicate result rows")

# Type conversion now that '\\N' placeholders were already replaced with NaN globally
results_df['grid'] = pd.to_numeric(results_df['grid'], errors='coerce')
results_df['positionOrder'] = pd.to_numeric(results_df['positionOrder'], errors='coerce')
results_df['points'] = pd.to_numeric(results_df['points'], errors='coerce')
results_df['laps'] = pd.to_numeric(results_df['laps'], errors='coerce')

# 'position' is null for anyone who didn't classify (DNF) — positionOrder always has a value
results_df['FinishPosition'] = pd.to_numeric(results_df['position'], errors='coerce')

results_df['GapSeconds'] = results_df['time'].apply(parse_gap_to_seconds)

# Join status category to derive IsDNF cleanly
results_df = results_df.merge(
    status_df[['statusId', 'status', 'StatusCategory']], on='statusId', how='left'
)

results_df['IsDNF'] = results_df['StatusCategory'] != 'Finished'
results_df['PositionsGained'] = results_df['grid'] - results_df['FinishPosition']
results_df['IsPodium'] = results_df['FinishPosition'] <= 3
results_df['IsWin'] = results_df['FinishPosition'] == 1

results_df[['raceId', 'driverId', 'grid', 'FinishPosition', 'points',
            'IsDNF', 'IsPodium', 'IsWin', 'PositionsGained']].head()


Dropped 91 duplicate result rows


,raceId,driverId,grid,FinishPosition,points,IsDNF,IsPodium,IsWin,PositionsGained
0,18,1,1.0,1.0,10.0,False,True,True,0.0
1,18,2,5.0,2.0,8.0,False,True,False,3.0
2,18,3,7.0,3.0,6.0,False,True,False,4.0
3,18,4,11.0,4.0,5.0,False,False,False,7.0
4,18,5,3.0,5.0,4.0,False,False,False,-2.0


## 9. Clean `qualifying`

In [17]:
qualifying_df = raw['qualifying'].copy()

for col in ['q1', 'q2', 'q3']:
    qualifying_df[f'{col}_ms'] = qualifying_df[col].apply(parse_lap_time_to_ms)

qualifying_df['EliminatedInQ1'] = qualifying_df['q2_ms'].isna()
qualifying_df['EliminatedInQ2'] = qualifying_df['q2_ms'].notna() & qualifying_df['q3_ms'].isna()

# Rename to match the FactQualifying schema in docs/Data_Dictionary.md
qualifying_df = qualifying_df.rename(columns={
    'position': 'QualifyingPosition',
    'q1_ms': 'Q1TimeMs',
    'q2_ms': 'Q2TimeMs',
    'q3_ms': 'Q3TimeMs',
})
qualifying_df['QualifyingPosition'] = pd.to_numeric(qualifying_df['QualifyingPosition'], errors='coerce')

qualifying_df.head()


,qualifyId,raceId,driverId,constructorId,number,QualifyingPosition,q1,q2,q3,Q1TimeMs,Q2TimeMs,Q3TimeMs,EliminatedInQ1,EliminatedInQ2
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714,86572.0,85187.0,86714.0,False,False
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869,86103.0,85315.0,86869.0,False,False
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079,85664.0,85452.0,87079.0,False,False
3,4,18,13,6,2,4,1:25.994,1:25.691,1:27.178,85994.0,85691.0,87178.0,False,False
4,5,18,2,2,3,5,1:25.960,1:25.518,1:27.236,85960.0,85518.0,87236.0,False,False


## 10. Clean `pit_stops`

In [19]:
pit_stops_df = raw['pit_stops'].copy()

pit_stops_df['DurationMs'] = pit_stops_df['duration'].apply(parse_duration_to_ms)
pit_stops_df['lap'] = pd.to_numeric(pit_stops_df['lap'], errors='coerce')
pit_stops_df['stop'] = pd.to_numeric(pit_stops_df['stop'], errors='coerce')

# Flag statistical outliers (>60s) — almost always non-strategic (drive-through penalty,
# damage repair)
pit_stops_df['IsAnomalousStop'] = pit_stops_df['DurationMs'] > 60_000

# Composite key for the relationship to FactPitStops
pit_stops_df['PitStopKey'] = (
    pit_stops_df['raceId'].astype(str) + '-' +
    pit_stops_df['driverId'].astype(str) + '-' +
    pit_stops_df['stop'].astype(str)
)

pit_stops_df.head()


,raceId,driverId,stop,lap,time,duration,milliseconds,DurationMs,IsAnomalousStop,PitStopKey
0,258,100,1,1,14:01:34,49.111,49111,49111.0,False,258-100-1
1,258,79,1,17,14:20:46,28.482,28482,28482.0,False,258-79-1
2,258,57,1,18,14:22:35,43.745,43745,43745.0,False,258-57-1
3,258,71,1,18,14:23:00,21.992,21992,21992.0,False,258-71-1
4,258,105,1,19,14:24:39,27.693,27693,27693.0,False,258-105-1


## 11. Clean `lap_times`

In [20]:
lap_times_df = raw['lap_times'].copy()

lap_times_df['LapTimeMs'] = lap_times_df['time'].apply(parse_lap_time_to_ms)
lap_times_df['position'] = pd.to_numeric(lap_times_df['position'], errors='coerce')
lap_times_df['lap'] = pd.to_numeric(lap_times_df['lap'], errors='coerce')

lap_times_df = lap_times_df[['raceId', 'driverId', 'lap', 'position', 'LapTimeMs']]

lap_times_df.head()


,raceId,driverId,lap,position,LapTimeMs
0,479,137,1,1,102085.0
1,479,137,2,2,96287.0
2,479,137,3,2,94627.0
3,479,137,4,2,94041.0
4,479,137,5,2,93699.0


## 12. Final Profiling Pass

Quick sanity check that cleaning didn't introduce new nulls or drop rows unexpectedly.


In [24]:
constructors_df = raw['constructors'].copy()

cleaned = {
    'drivers': drivers_df,
    'constructors': constructors_df,
    'circuits': circuits_df,
    'races': races_enriched,
    'results': results_df,
    'qualifying': qualifying_df,
    'pit_stops': pit_stops_df,
    'lap_times': lap_times_df,
    'status': status_df,
}

for name, df in cleaned.items():
    print(f"{name:15s} -> {df.shape[0]:>7,} rows, {df.shape[1]:>2} cols, "
          f"nulls in key cols OK: {df.isnull().sum().sum()} total nulls across all cols")


drivers         ->     865 rows, 13 cols, nulls in key cols OK: 1559 total nulls across all cols
constructors    ->     214 rows,  5 cols, nulls in key cols OK: 0 total nulls across all cols
circuits        ->      78 rows, 10 cols, nulls in key cols OK: 0 total nulls across all cols
races           ->   1,171 rows, 24 cols, nulls in key cols OK: 11477 total nulls across all cols
results         ->  27,323 rows, 26 cols, nulls in key cols OK: 166680 total nulls across all cols
qualifying      ->  11,146 rows, 14 cols, nulls in key cols OK: 24372 total nulls across all cols
pit_stops       ->  22,424 rows, 10 cols, nulls in key cols OK: 773 total nulls across all cols
lap_times       -> 875,093 rows,  5 cols, nulls in key cols OK: 309 total nulls across all cols
status          ->     140 rows,  3 cols, nulls in key cols OK: 0 total nulls across all cols


## 13. Export to Parquet

Parquet (not CSV) for the Power BI load — smaller, faster to refresh, and preserves
data types, avoiding the "numbers stored as text" class of bug entirely.


In [25]:
for name, df in cleaned.items():
    out_path = OUT_DIR / f'{name}.parquet'
    df.to_parquet(out_path, index=False)
    print(f"Wrote {out_path}  ({df.shape[0]:,} rows)")

print("\nDone — ready for feature_engineering.ipynb")


Wrote ..\data\processed\drivers.parquet  (865 rows)
Wrote ..\data\processed\constructors.parquet  (214 rows)
Wrote ..\data\processed\circuits.parquet  (78 rows)
Wrote ..\data\processed\races.parquet  (1,171 rows)
Wrote ..\data\processed\results.parquet  (27,323 rows)
Wrote ..\data\processed\qualifying.parquet  (11,146 rows)
Wrote ..\data\processed\pit_stops.parquet  (22,424 rows)
Wrote ..\data\processed\lap_times.parquet  (875,093 rows)
Wrote ..\data\processed\status.parquet  (140 rows)

Done — ready for feature_engineering.ipynb
